# M2 — วัดความซื่อตรงของชั้นอธิบายด้วย LLM (Typhoon2 8B)

วัด entity preservation, technique grounding และ claim-level support บน alert 100 เคส
ที่สุ่มแบบ stratified จาก **unified test split**

## ทำไมต้องใช้ `!uv run` ไม่ใช่ kernel นี้ตรงๆ

สคริปต์ต้องใช้สอง interpreter ที่ติดตั้งคนละชุด:

| stage | ต้องการ | อยู่ที่ |
|---|---|---|
| score RF / IF / AE + RAG | pandas, torch, sentence-transformers | Anaconda |
| เรียก LLM | LangChain | `.venv` (uv) |

`!uv run` เป็นคำสั่ง shell ดังนั้น **kernel ของ notebook นี้ไม่เกี่ยว** — สคริปต์จะ shell out
ไปเรียก Anaconda เองสำหรับ stage แรก ถ้าพยายาม import ทั้งก้อนใน kernel Anaconda จะพังที่ LangChain

> **ลำดับที่แนะนำ: ให้ Cell 4 (ทดลอง 5 เคส) ผ่านก่อนไป Cell 5**
> ถ้า prompt หรือ guard มีปัญหาจะรู้ใน ~2 นาที ไม่ใช่เสียเวลา 45 นาที

## Cell 1 — ตั้ง working directory และเช็คว่ามี alert set แล้วหรือยัง

In [ ]:
import os, pathlib

os.chdir(r"E:\thesis")
os.environ["PYTHONIOENCODING"] = "utf-8"

print("cwd:", os.getcwd())
alerts = pathlib.Path("reports/llm_faithfulness_alerts.jsonl")
if alerts.exists():
    n = sum(1 for _ in alerts.open(encoding="utf-8"))
    print(f"alert set พร้อมแล้ว: {n} เคส (stage alerts จะถูกข้าม)")
else:
    print("ยังไม่มี alert set — สคริปต์จะสร้างให้อัตโนมัติ (ใช้เวลาเพิ่ม ~2 นาที)")

## Cell 2 — เปิด Ollama

ถ้า Ollama ไม่รัน stage LLM จะพังทันที cell นี้เช็คก่อนแล้วเปิดให้ถ้ายังไม่เปิด
(จะเด้งหน้าต่าง console ใหม่ขึ้นมา ปล่อยไว้ อย่าปิด)

In [ ]:
import subprocess, time, urllib.request

OLLAMA = r"C:\Users\devMrX\AppData\Local\Programs\Ollama\ollama.exe"


def ollama_up(timeout=3):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=timeout)
        return True
    except Exception:
        return False


if ollama_up():
    print("Ollama รันอยู่แล้ว")
else:
    subprocess.Popen([OLLAMA, "serve"], creationflags=subprocess.CREATE_NEW_CONSOLE)
    for _ in range(20):
        time.sleep(2)
        if ollama_up():
            print("เปิด Ollama สำเร็จ")
            break
    else:
        print("เปิดไม่ขึ้น — ลองเปิด Ollama จาก Start Menu ด้วยมือ แล้วรัน cell นี้ซ้ำ")

## Cell 3 — ยืนยันว่ามีโมเดล Typhoon2

ต้องเห็น `scb10x/llama3.1-typhoon2-8b-instruct` ในรายการ

In [ ]:
!"C:\Users\devMrX\AppData\Local\Programs\Ollama\ollama.exe" list

# ถ้าไม่เห็น scb10x/llama3.1-typhoon2-8b-instruct ให้เอา # ออกแล้วรันซ้ำ (ดาวน์โหลด ~5 GB)
# !"C:\Users\devMrX\AppData\Local\Programs\Ollama\ollama.exe" pull scb10x/llama3.1-typhoon2-8b-instruct

## Cell 4 — ทดลอง 5 เคส (ทำก่อน)

ใช้เวลา ~2 นาที

ถ้าเจอ `ValueError` เรื่อง foreign entity **นั่นคือ guard ทำงานถูกต้อง ไม่ใช่บั๊ก** —
มันปฏิเสธ output ที่อ้าง IP / hostname / technique id ที่ไม่มีอยู่ใน alert จริง
เพราะคำอธิบายที่ชี้ไปเครื่องผิดแย่กว่าการไม่มีคำอธิบาย

In [ ]:
!uv run python -m src.evaluation.eval_llm_faithfulness --limit 5

## Cell 5 — รันเต็ม 100 เคส

ใช้เวลา ~45 นาที · **มี per-alert checkpoint** ถ้า notebook หลุดกลางทาง
รัน cell นี้ซ้ำจะต่อจากเคสที่ค้าง ไม่เริ่มใหม่

`!` ไม่ stream output สด จะเห็น log ตอน cell จบ

In [ ]:
!uv run python -m src.evaluation.eval_llm_faithfulness

## Cell 6 — อ่านรายงานผล

In [ ]:
import pathlib

rep = pathlib.Path("reports/llm_faithfulness_report.txt")
if rep.exists():
    print(rep.read_text(encoding="utf-8")[:4000])
else:
    print("ยังไม่มีรายงาน — รัน Cell 4 หรือ 5 ก่อน")

## Cell 7 — คำนวณ metric ใหม่จาก raw ที่มีอยู่ (ไม่เรียก LLM ซ้ำ)

ใช้เมื่อแก้เกณฑ์การให้คะแนนแล้วอยากคิดใหม่ โดยไม่เสียเวลาเรียกโมเดลอีก 45 นาที

In [ ]:
!uv run python -m src.evaluation.eval_llm_faithfulness --rescore

## Cell 8 — ตรวจว่าผลมาจาก pipeline ที่ deploy จริง

ตัวเลข M2 ชุดก่อนหน้านี้วัดบน pipeline ที่ถูก supersede แล้ว cell นี้ยืนยันว่าชุดใหม่ไม่ซ้ำรอย

In [ ]:
import json, pathlib
from collections import Counter

raw = pathlib.Path("reports/llm_faithfulness_raw.jsonl")
if not raw.exists():
    print("ยังไม่มีผล — รัน Cell 4 หรือ 5 ก่อน")
else:
    rows = [json.loads(l) for l in raw.open(encoding="utf-8") if l.strip()]
    used = Counter(r.get("alert_context", {}).get("model_used") for r in rows)
    print(f"เคสที่วัดแล้ว: {len(rows)}")
    print("model_used:", dict(used))
    stale = {k for k in used if k not in ("random_forest", "if_ae_fallback")}
    if stale:
        print(f"!! พบค่าที่ไม่ควรมีอีกแล้ว: {stale} — แปลว่ามีของเก่าค้างอยู่")
    else:
        print("ตรงกับกฎที่ deploy (random_forest / if_ae_fallback) ถูกต้อง")

---
## หมายเหตุสำคัญ

- **ตัวเลข M2 ชุดเก่าอ้างไม่ได้** เพราะวัดบน pipeline ที่ไม่ได้ deploy แล้ว
  (RF 80 features + behavioral_v2 IF/AE + row-hash join ที่ unified population ยกเลิกไป)
  ผลจาก notebook นี้คือชุดที่อ้างได้
- **`model_used` ควรเห็นแค่ `random_forest` กับ `if_ae_fallback`** — ถ้าเห็น
  `autoencoder_v2` แปลว่ามีของเก่าค้างอยู่
- **อัตราส่วน branch ที่คาด** ≈ 88 / 12 จาก 100 เคส (ตรวจแล้วตอนสร้าง alert set ใหม่)
- **ผลที่ได้ยังไม่ใช่การตรวจคุณภาพเชิงภาษา** — M2 วัดเชิงกลไก (entity ตรงไหม,
  technique มีหลักฐานรองรับไหม) ส่วนการตัดสินว่าคำอธิบายภาษาไทย *ใช้งานได้จริง* หรือไม่
  ต้องมีคนอ่าน ซึ่งเป็นงานถัดไปและสคริปต์แทนไม่ได้